# 02 · Baseline IDS Models (Stage 2 — Experiment E1)
Implements **Section 8.2** and experiment **E1** of the project plan:

| Model | Design |
|---|---|
| Random Forest | 150 trees, default settings (`n_jobs=-1`) |
| MLP | Dense 128-64-32, ReLU, Dropout 0.3 |
| 1D-CNN | Conv1D(32) → MaxPool → Conv1D(64) → MaxPool → Dense 64 → output |
| LSTM | 1 LSTM layer (64 units) over the feature sequence → Dense output |

All DL models use the same training settings: **Adam (lr 1e-3)**, **cross-entropy loss**, batch 512, up to 40 epochs, **early stopping on validation loss** (patience 5, best weights restored).
Every model is trained with **3 seeds (42, 102, 2025)**. We report **mean ± std** of Accuracy, Precision, Recall, F1 (macro and per class), FPR and the confusion matrices on the **original test set**.

**Shared code.** Model definitions, the training loop and the metric functions live in `ids_common.py`, which this notebook writes to `MyDrive/IDS_Polymorphic_Project/code/`. Notebooks 04 and 05 import the same file, so E2–E4 use identical models, hyperparameters and metrics.

**Runtime.** Use a GPU runtime (*Runtime → Change runtime type → T4 GPU*). The LSTM is the slowest model; the full 12-run loop takes about 20–40 min on a T4. **Resumable:** finished runs are saved to Drive and skipped on re-run, so a Colab disconnect costs at most one run.

**Prerequisite:** `01_preprocessing.ipynb` has been run.

## 1 · Mount Drive and load the project configuration

In [ ]:
import os, sys, json, time, gc
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/IDS_Polymorphic_Project'
else:
    BASE_DIR = os.path.abspath('./IDS_Polymorphic_Project')

with open(os.path.join(BASE_DIR, 'config', 'project_config.json')) as fh:
    PCFG = json.load(fh)
with open(os.path.join(BASE_DIR, 'config', 'feature_config.json')) as fh:
    FCFG = json.load(fh)

DIRS = {k: v.replace(PCFG['dirs']['base'], BASE_DIR) for k, v in PCFG['dirs'].items()}
DIRS['code'] = os.path.join(BASE_DIR, 'code')
DIRS['baseline_models'] = os.path.join(DIRS['models'], 'baseline')
DIRS['runs'] = os.path.join(DIRS['results'], '02_runs')
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

CLASS_NAMES = PCFG['class_names']; BENIGN_ID = PCFG['benign_id']; MODEL_SEEDS = PCFG['model_seeds']
print('Classes:', CLASS_NAMES, '| benign id:', BENIGN_ID, '| seeds:', MODEL_SEEDS)
print('Features:', len(FCFG['feature_names']))

## 2 · Write the shared module `ids_common.py`
Written to the Colab working directory and copied to `code/` on Drive (notebooks 04/05 import it from there).

In [ ]:
%%writefile ids_common.py
"""Shared models, training, prediction and metric utilities for the IDS polymorphic-robustness project.
Imported by notebooks 02, 04 and 05 so that every experiment uses identical settings."""
import os, json, time, random
import numpy as np
import torch
import torch.nn as nn
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix)

MODEL_NAMES = ['RandomForest', 'MLP', 'CNN1D', 'LSTM']
DL_MODELS = ['MLP', 'CNN1D', 'LSTM']

RF_HPARAMS = {'n_estimators': 150, 'n_jobs': -1}
DL_HPARAMS = {'lr': 1e-3, 'batch_size': 512, 'max_epochs': 40, 'patience': 5, 'min_delta': 1e-4,
              'eval_batch_size': 4096}


# ------------------------------------------------------------------ reproducibility
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)


def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# ------------------------------------------------------------------ models
class MLP(nn.Module):
    def __init__(self, n_features, n_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, n_classes))

    def forward(self, x):
        return self.net(x)


class CNN1D(nn.Module):
    def __init__(self, n_features, n_classes, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool1d(2))
        flat = 64 * (n_features // 4)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(flat, 64), nn.ReLU(), nn.Dropout(dropout),
                                  nn.Linear(64, n_classes))

    def forward(self, x):
        return self.head(self.features(x.unsqueeze(1)))       # (B, F) -> (B, 1, F)


class LSTMNet(nn.Module):
    def __init__(self, n_features, n_classes, hidden=64, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, num_layers=1, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, n_classes))

    def forward(self, x):
        _, (h, _) = self.lstm(x.unsqueeze(-1))                # (B, F) -> (B, F, 1); h: (1, B, H)
        return self.head(h[-1])


def build_dl_model(name, n_features, n_classes):
    if name == 'MLP':
        return MLP(n_features, n_classes)
    if name == 'CNN1D':
        return CNN1D(n_features, n_classes)
    if name == 'LSTM':
        return LSTMNet(n_features, n_classes)
    raise ValueError(f'unknown DL model {name}')


# ------------------------------------------------------------------ training
def train_rf(X, y, seed):
    rf = RandomForestClassifier(random_state=seed, **RF_HPARAMS)
    t0 = time.time()
    rf.fit(X, y)
    return rf, {'train_time_s': time.time() - t0}


@torch.no_grad()
def _eval_loss(model, X, y, criterion, bs):
    model.eval()
    total, n = 0.0, 0
    for i in range(0, len(X), bs):
        xb, yb = X[i:i + bs], y[i:i + bs]
        total += criterion(model(xb), yb).item() * len(xb); n += len(xb)
    return total / n


def train_dl(name, X_tr, y_tr, X_va, y_va, seed, n_classes, device=None, class_weights=None, verbose=True):
    """Adam + cross-entropy, early stopping on validation loss with best-weight restore."""
    device = device or get_device()
    set_seed(seed)
    hp = DL_HPARAMS
    Xtr = torch.as_tensor(X_tr, dtype=torch.float32, device=device)
    ytr = torch.as_tensor(y_tr, dtype=torch.long, device=device)
    Xva = torch.as_tensor(X_va, dtype=torch.float32, device=device)
    yva = torch.as_tensor(y_va, dtype=torch.long, device=device)

    model = build_dl_model(name, Xtr.shape[1], n_classes).to(device)
    w = None
    if class_weights is not None:
        w = torch.as_tensor([class_weights[i] for i in range(n_classes)], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=w)
    eval_criterion = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(model.parameters(), lr=hp['lr'])
    gen = torch.Generator(device='cpu').manual_seed(seed)

    best_loss, best_state, best_epoch, wait = float('inf'), None, 0, 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    t0 = time.time()
    for epoch in range(1, hp['max_epochs'] + 1):
        model.train()
        perm = torch.randperm(len(Xtr), generator=gen).to(device)
        run, n = 0.0, 0
        for i in range(0, len(Xtr), hp['batch_size']):
            idx = perm[i:i + hp['batch_size']]
            xb, yb = Xtr[idx], ytr[idx]
            opt.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
            run += loss.item() * len(idx); n += len(idx)
        tr_loss = run / n
        va_loss = _eval_loss(model, Xva, yva, eval_criterion, hp['eval_batch_size'])
        va_pred = predict_dl(model, Xva, device=device)
        va_acc = float((va_pred == y_va).mean())
        history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss); history['val_acc'].append(va_acc)
        improved = va_loss < best_loss - hp['min_delta']
        if improved:
            best_loss, best_epoch, wait = va_loss, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        if verbose:
            print(f'    [{name} seed {seed}] epoch {epoch:02d}  train {tr_loss:.4f}  val {va_loss:.4f}  '
                  f'val_acc {va_acc:.4f}{"  *" if improved else ""}')
        if wait >= hp['patience']:
            break
    model.load_state_dict(best_state)
    info = {'train_time_s': time.time() - t0, 'best_epoch': best_epoch, 'epochs_run': epoch,
            'best_val_loss': best_loss, 'history': history}
    del Xtr, ytr, Xva, yva
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return model, info


@torch.no_grad()
def predict_proba_dl(model, X, device=None, bs=None):
    device = device or get_device()
    bs = bs or DL_HPARAMS['eval_batch_size']
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        xb = X[i:i + bs]
        if not torch.is_tensor(xb):
            xb = torch.as_tensor(np.asarray(xb), dtype=torch.float32)
        out.append(torch.softmax(model(xb.to(device)), dim=1).cpu().numpy())
    return np.concatenate(out, axis=0)


def predict_dl(model, X, device=None, bs=None):
    return predict_proba_dl(model, X, device, bs).argmax(axis=1)


def predict(model_name, model, X, device=None):
    if model_name == 'RandomForest':
        return model.predict(X)
    return predict_dl(model, X, device=device)


# ------------------------------------------------------------------ persistence
def model_path(model_dir, model_name, seed):
    ext = 'joblib' if model_name == 'RandomForest' else 'pt'
    return os.path.join(model_dir, f'{model_name}_seed{seed}.{ext}')


def save_model(model_name, model, path, n_features=None, n_classes=None):
    if model_name == 'RandomForest':
        joblib.dump(model, path, compress=3)
    else:
        torch.save({'state_dict': model.state_dict(), 'model_name': model_name,
                    'n_features': n_features, 'n_classes': n_classes}, path)


def load_model(model_name, path, device=None):
    if model_name == 'RandomForest':
        return joblib.load(path)
    device = device or get_device()
    ckpt = torch.load(path, map_location=device)
    m = build_dl_model(model_name, ckpt['n_features'], ckpt['n_classes']).to(device)
    m.load_state_dict(ckpt['state_dict'])
    m.eval()
    return m


# ------------------------------------------------------------------ metrics
def compute_metrics(y_true, y_pred, class_names, benign_id=0):
    """Multi-class metrics + binary (attack vs benign) FPR / detection rate."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    labels = list(range(len(class_names)))
    p_c, r_c, f_c, s_c = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    p_m, r_m, f_m, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average='macro', zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average='weighted', zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    true_att = y_true != benign_id
    pred_att = y_pred != benign_id
    tn = int((~true_att & ~pred_att).sum()); fp = int((~true_att & pred_att).sum())
    fn = int((true_att & ~pred_att).sum()); tp = int((true_att & pred_att).sum())
    m = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(p_m), 'recall_macro': float(r_m), 'f1_macro': float(f_m),
        'precision_weighted': float(p_w), 'recall_weighted': float(r_w), 'f1_weighted': float(f_w),
        'fpr': fp / (fp + tn) if (fp + tn) else float('nan'),               # benign flagged as any attack
        'detection_rate': tp / (tp + fn) if (tp + fn) else float('nan'),    # binary attack recall
        'evasion_rate': fn / (tp + fn) if (tp + fn) else float('nan'),      # attacks predicted Benign
        'n_samples': int(len(y_true)),
    }
    for i, c in enumerate(class_names):
        m[f'precision_{c}'] = float(p_c[i]); m[f'recall_{c}'] = float(r_c[i])
        m[f'f1_{c}'] = float(f_c[i]); m[f'support_{c}'] = int(s_c[i])
    return m, cm


def attack_set_metrics(y_true, y_pred, class_names, benign_id=0):
    """For sets that contain ONLY attack flows (LLM / scripted variants):
    per-class recall (correct attack class), detection (any attack label) and evasion (labelled Benign)."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    out = {'n_samples': int(len(y_true)),
           'recall_overall': float((y_true == y_pred).mean()) if len(y_true) else float('nan'),
           'detection_rate': float((y_pred != benign_id).mean()) if len(y_true) else float('nan'),
           'evasion_rate': float((y_pred == benign_id).mean()) if len(y_true) else float('nan')}
    for i, c in enumerate(class_names):
        if i == benign_id:
            continue
        mask = y_true == i
        n = int(mask.sum())
        out[f'n_{c}'] = n
        out[f'recall_{c}'] = float((y_pred[mask] == i).mean()) if n else float('nan')
        out[f'detection_{c}'] = float((y_pred[mask] != benign_id).mean()) if n else float('nan')
        out[f'evasion_{c}'] = float((y_pred[mask] == benign_id).mean()) if n else float('nan')
    return out


def mean_std_table(df, group_cols, metric_cols, decimals=4):
    """Return (numeric mean/std frame, pretty 'mean ± std' string frame)."""
    g = df.groupby(group_cols)[metric_cols]
    mean, std = g.mean(), g.std(ddof=1).fillna(0.0)
    pretty = mean.copy().astype(object)
    for c in metric_cols:
        pretty[c] = [f'{a:.{decimals}f} ± {b:.{decimals}f}' for a, b in zip(mean[c], std[c])]
    num = mean.add_suffix('_mean').join(std.add_suffix('_std'))
    return num, pretty

In [ ]:
import shutil
shutil.copy('ids_common.py', os.path.join(DIRS['code'], 'ids_common.py'))
sys.path.insert(0, os.getcwd())
import importlib, ids_common
importlib.reload(ids_common)
from ids_common import *
print('ids_common copied to', DIRS['code'])

## 3 · Imports, device and data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, sklearn

DEVICE = get_device()
print('torch', torch.__version__, '| sklearn', sklearn.__version__, '| device:', DEVICE,
      f'({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else '(no GPU: LSTM will be slow)')

P = DIRS['processed']
X_train = np.load(os.path.join(P, 'X_train.npy')); y_train = np.load(os.path.join(P, 'y_train.npy'))
X_val   = np.load(os.path.join(P, 'X_val.npy'));   y_val   = np.load(os.path.join(P, 'y_val.npy'))
X_test  = np.load(os.path.join(P, 'X_test.npy'));  y_test  = np.load(os.path.join(P, 'y_test.npy'))
N_FEATURES, N_CLASSES = X_train.shape[1], len(CLASS_NAMES)
for n, X, y in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    print(f'{n:<5} X={X.shape} dtype={X.dtype} | class counts:', dict(zip(CLASS_NAMES, np.bincount(y, minlength=N_CLASSES))))

## 4 · Experiment settings
`USE_CLASS_WEIGHTS = False` by default: train/val are already balanced in notebook 01. Set it to `True` to weight the loss with the balanced class weights saved in `project_config.json`. The setting is saved, and notebook 05 reuses it.

In [ ]:
MODELS_TO_RUN = MODEL_NAMES            # ['RandomForest', 'MLP', 'CNN1D', 'LSTM']
USE_CLASS_WEIGHTS = False
FORCE_RETRAIN = False                  # True = ignore saved runs and retrain everything
VERBOSE_EPOCHS = True

CLASS_WEIGHTS = {int(k): v for k, v in PCFG['class_weights_balanced'].items()} if USE_CLASS_WEIGHTS else None
exp_settings = {'models': MODELS_TO_RUN, 'seeds': MODEL_SEEDS, 'use_class_weights': USE_CLASS_WEIGHTS,
                'rf_hparams': RF_HPARAMS, 'dl_hparams': DL_HPARAMS, 'n_features': N_FEATURES,
                'architectures': {'MLP': 'Dense 128-64-32 ReLU, dropout 0.3',
                                  'CNN1D': 'Conv1d(1->32,k3)+MaxPool2, Conv1d(32->64,k3)+MaxPool2, Dense 64, dropout 0.3',
                                  'LSTM': 'LSTM(64) over features as sequence, dropout 0.2, Dense',
                                  'RandomForest': '150 trees, sklearn defaults'}}
with open(os.path.join(DIRS['config'], 'training_config.json'), 'w') as fh:
    json.dump(exp_settings, fh, indent=2)
print(json.dumps(exp_settings, indent=2))

## 5 · E1 training & evaluation loop (4 models × 3 seeds)
For each run: train → save the model to `models/baseline/` → evaluate on the original test set → save metrics, confusion matrix and history to `results/02_runs/`. Finished runs are loaded instead of retrained.

In [ ]:
def run_files(model_name, seed):
    stem = os.path.join(DIRS['runs'], f'{model_name}_seed{seed}')
    return stem + '_metrics.json', stem + '_cm.npy', stem + '_history.json'

all_runs = []
t_start = time.time()
for seed in MODEL_SEEDS:
    for name in MODELS_TO_RUN:
        mpath = model_path(DIRS['baseline_models'], name, seed)
        f_met, f_cm, f_hist = run_files(name, seed)
        if not FORCE_RETRAIN and all(os.path.exists(p) for p in (mpath, f_met, f_cm)):
            with open(f_met) as fh:
                all_runs.append(json.load(fh))
            print(f'[skip] {name} seed {seed}: loaded saved run')
            continue

        print(f'\n=== {name} | seed {seed} ===')
        if name == 'RandomForest':
            model, info = train_rf(X_train, y_train, seed)
        else:
            model, info = train_dl(name, X_train, y_train, X_val, y_val, seed, N_CLASSES,
                                   device=DEVICE, class_weights=CLASS_WEIGHTS, verbose=VERBOSE_EPOCHS)
        save_model(name, model, mpath, N_FEATURES, N_CLASSES)

        t0 = time.time()
        y_pred = predict(name, model, X_test, device=DEVICE)
        infer_s = time.time() - t0
        metrics, cm = compute_metrics(y_test, y_pred, CLASS_NAMES, BENIGN_ID)
        metrics.update({'model': name, 'seed': seed, 'train_time_s': info['train_time_s'], 'inference_time_s': infer_s,
                        'best_epoch': info.get('best_epoch'), 'epochs_run': info.get('epochs_run')})
        with open(f_met, 'w') as fh:
            json.dump(metrics, fh, indent=2)
        np.save(f_cm, cm)
        if 'history' in info:
            with open(f_hist, 'w') as fh:
                json.dump(info['history'], fh)
        all_runs.append(metrics)
        print(f'  -> acc {metrics["accuracy"]:.4f} | F1-macro {metrics["f1_macro"]:.4f} | '
              f'recall-macro {metrics["recall_macro"]:.4f} | FPR {metrics["fpr"]:.4f} | '
              f'train {info["train_time_s"]:.0f}s')
        del model; gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

print(f'\nAll runs ready ({len(all_runs)}) in {(time.time() - t_start) / 60:.1f} min')

## 6 · Results: mean ± std over seeds

In [ ]:
runs_df = pd.DataFrame(all_runs)
runs_df['model'] = pd.Categorical(runs_df['model'], categories=MODELS_TO_RUN, ordered=True)
runs_df = runs_df.sort_values(['model', 'seed']).reset_index(drop=True)
runs_df.to_csv(os.path.join(DIRS['results'], '02_E1_runs_all_seeds.csv'), index=False)

MAIN = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'fpr', 'detection_rate', 'evasion_rate']
num_main, pretty_main = mean_std_table(runs_df, 'model', MAIN)
num_main.to_csv(os.path.join(DIRS['results'], '02_E1_summary_mean_std.csv'))
pretty_main.to_csv(os.path.join(DIRS['results'], '02_E1_summary_pretty.csv'))
print('E1 — original test set (mean ± std over seeds', MODEL_SEEDS, ')')
display(pretty_main)

In [ ]:
rows = []
for c in CLASS_NAMES:
    for metric in ['precision', 'recall', 'f1']:
        col = f'{metric}_{c}'
        g = runs_df.groupby('model', observed=True)[col]
        for m, mean, std in zip(g.mean().index, g.mean().values, g.std(ddof=1).fillna(0).values):
            rows.append({'model': m, 'class': c, 'metric': metric, 'mean': mean, 'std': std})
per_class = pd.DataFrame(rows)
per_class.to_csv(os.path.join(DIRS['results'], '02_E1_per_class_mean_std.csv'), index=False)
per_class_pretty = per_class.assign(value=lambda d: d['mean'].map('{:.4f}'.format) + ' ± ' + d['std'].map('{:.4f}'.format)) \
    .pivot_table(index=['model', 'class'], columns='metric', values='value', aggfunc='first', observed=True)[['precision', 'recall', 'f1']]
per_class_pretty.to_csv(os.path.join(DIRS['results'], '02_E1_per_class_pretty.csv'))
print(per_class_pretty.to_string())

# original per-class recall per model (mean over seeds) -> reference for Recall Drop in notebook 04
ref = runs_df.groupby('model', observed=True)[[f'recall_{c}' for c in CLASS_NAMES if c != CLASS_NAMES[BENIGN_ID]] + ['detection_rate']].mean()
ref.to_csv(os.path.join(DIRS['results'], '02_E1_reference_recall.csv'))

## 7 · Confusion matrices (mean over seeds, row-normalised)

In [ ]:
fig, axes = plt.subplots(1, len(MODELS_TO_RUN), figsize=(4.6 * len(MODELS_TO_RUN), 4.4))
axes = np.atleast_1d(axes)
cm_store = {}
for ax, name in zip(axes, MODELS_TO_RUN):
    cms = np.stack([np.load(run_files(name, s)[1]) for s in MODEL_SEEDS]).astype(float)
    cm_sum = cms.sum(axis=0)
    cm_norm = cm_sum / cm_sum.sum(axis=1, keepdims=True).clip(min=1)
    cm_store[name] = {'counts_per_seed': cms.astype(int).tolist(), 'mean_counts': cms.mean(axis=0).tolist(),
                      'row_normalised': cm_norm.tolist()}
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            ax.text(j, i, f'{cm_norm[i, j]:.3f}', ha='center', va='center', fontsize=8,
                    color='white' if cm_norm[i, j] > 0.6 else 'black')
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=30, fontsize=8)
    ax.set_yticks(range(len(CLASS_NAMES))); ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_xlabel('predicted'); ax.set_title(name)
axes[0].set_ylabel('true')
plt.suptitle('E1 — confusion matrices on the original test set (3 seeds pooled, row-normalised)')
plt.tight_layout()
plt.savefig(os.path.join(DIRS['results'], '02_E1_confusion_matrices.png'), dpi=150)
plt.show()
with open(os.path.join(DIRS['results'], '02_E1_confusion_matrices.json'), 'w') as fh:
    json.dump({'class_names': CLASS_NAMES, 'matrices': cm_store}, fh)

## 8 · Charts: macro metrics and per-class recall

In [ ]:
colors = ['#2f6db5', '#d1495b', '#edae49', '#00798c']
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

metrics_plot = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
x = np.arange(len(metrics_plot)); w = 0.8 / len(MODELS_TO_RUN)
for k, name in enumerate(MODELS_TO_RUN):
    sub = runs_df[runs_df['model'] == name]
    axes[0].bar(x + k * w - 0.4 + w / 2, sub[metrics_plot].mean(), w, yerr=sub[metrics_plot].std(ddof=1).fillna(0),
                capsize=3, label=name, color=colors[k % 4])
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics_plot)
lo = max(0.0, runs_df[metrics_plot].min().min() - 0.05)
axes[0].set_ylim(lo, 1.0); axes[0].set_title('Macro metrics (mean ± std over 3 seeds)'); axes[0].legend(frameon=False, fontsize=8)

x2 = np.arange(len(CLASS_NAMES))
for k, name in enumerate(MODELS_TO_RUN):
    sub = runs_df[runs_df['model'] == name]
    cols = [f'recall_{c}' for c in CLASS_NAMES]
    axes[1].bar(x2 + k * w - 0.4 + w / 2, sub[cols].mean(), w, yerr=sub[cols].std(ddof=1).fillna(0),
                capsize=3, label=name, color=colors[k % 4])
axes[1].set_xticks(x2); axes[1].set_xticklabels(CLASS_NAMES)
lo2 = max(0.0, runs_df[[f'recall_{c}' for c in CLASS_NAMES]].min().min() - 0.05)
axes[1].set_ylim(lo2, 1.0); axes[1].set_title('Per-class recall on the original test set')
for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(DIRS['results'], '02_E1_metrics_bars.png'), dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
fp = runs_df.groupby('model', observed=True)['fpr'].agg(['mean', 'std'])
ax.bar(fp.index.astype(str), fp['mean'], yerr=fp['std'].fillna(0), capsize=4, color=colors[:len(fp)])
ax.set_ylabel('FPR (benign flagged as attack)'); ax.set_title('False-positive rate on the realistic test set')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(os.path.join(DIRS['results'], '02_E1_fpr.png'), dpi=150); plt.show()

## 9 · Training curves (DL models, validation loss with early-stopping point)

In [ ]:
dl_names = [m for m in MODELS_TO_RUN if m in DL_MODELS]
if dl_names:
    fig, axes = plt.subplots(1, len(dl_names), figsize=(5.2 * len(dl_names), 3.8))
    axes = np.atleast_1d(axes)
    for ax, name in zip(axes, dl_names):
        for s, col in zip(MODEL_SEEDS, ['#2f6db5', '#d1495b', '#00798c']):
            hp = run_files(name, s)[2]
            if not os.path.exists(hp):
                continue
            h = json.load(open(hp))
            ep = np.arange(1, len(h['val_loss']) + 1)
            ax.plot(ep, h['train_loss'], '--', color=col, alpha=0.6, lw=1)
            ax.plot(ep, h['val_loss'], '-', color=col, lw=1.6, label=f'seed {s}')
            best = int(np.argmin(h['val_loss']))
            ax.scatter(best + 1, h['val_loss'][best], color=col, zorder=5, s=25)
        ax.set_yscale('log'); ax.set_title(f'{name} (dashed = train, solid = val)')
        ax.set_xlabel('epoch'); ax.legend(frameon=False, fontsize=8); ax.spines[['top', 'right']].set_visible(False)
    axes[0].set_ylabel('cross-entropy loss')
    plt.tight_layout(); plt.savefig(os.path.join(DIRS['results'], '02_E1_training_curves.png'), dpi=150); plt.show()

eff = runs_df.groupby('model', observed=True)[['train_time_s', 'inference_time_s', 'epochs_run', 'best_epoch']].mean().round(2)
eff.to_csv(os.path.join(DIRS['results'], '02_E1_efficiency.csv'))
print(eff)

## 10 · Verify saved models reload correctly

In [ ]:
check = []
for name in MODELS_TO_RUN:
    seed = MODEL_SEEDS[0]
    m = load_model(name, model_path(DIRS['baseline_models'], name, seed), device=DEVICE)
    y_pred = predict(name, m, X_test, device=DEVICE)
    f1 = compute_metrics(y_test, y_pred, CLASS_NAMES, BENIGN_ID)[0]['f1_macro']
    saved = runs_df[(runs_df['model'] == name) & (runs_df['seed'] == seed)]['f1_macro'].iloc[0]
    check.append((name, round(f1, 6), round(saved, 6), abs(f1 - saved) < 1e-6))
    del m
print(pd.DataFrame(check, columns=['model', 'reloaded_f1_macro', 'saved_f1_macro', 'match']))
assert all(c[3] for c in check), 'Reloaded model does not reproduce saved metrics!'
print('\nSaved models:')
for f in sorted(os.listdir(DIRS['baseline_models'])):
    print(f'  models/baseline/{f}  ({os.path.getsize(os.path.join(DIRS["baseline_models"], f)) / 1024**2:.1f} MB)')

## 11 · Summary

In [ ]:
print('=' * 72); print('STAGE 2 / EXPERIMENT E1 COMPLETE'); print('=' * 72)
print(pretty_main[['accuracy', 'f1_macro', 'recall_macro', 'fpr']].to_string())
best = num_main['f1_macro_mean'].idxmax()
print(f'\nBest macro-F1: {best} ({num_main.loc[best, "f1_macro_mean"]:.4f})')
print('\nResult files:')
for f in sorted(os.listdir(DIRS['results'])):
    if f.startswith('02_'):
        print('  results/' + f)
print('\nNext: run 03_llm_variant_generation.ipynb')

### Notes for the report
- The **test set is benign-heavy** (realistic mix), so accuracy is dominated by benign traffic; **macro-F1**, **per-class recall** and **FPR** are the meaningful numbers.
- `detection_rate` / `evasion_rate` here are binary (attack vs benign) values on the original test attacks. They are the reference points for **Recall Drop** and **Evasion Rate** in notebook 04 (`02_E1_reference_recall.csv`).
- Very high scores (≥ 0.99) are expected on CIC-IDS2017 even after de-duplication. The research question is how much of this survives on polymorphic variants.